# Surprise-dimension discovery

Steps 1-2 of the plan:

1. **Split** 2026Q3 events that have both a preview and a realized return chronologically: the earliest 80% are **train**, the latest 20% **test**. Then sample 500 **discovery** events from train only, stratified by next-day abnormal return (`car1`) decile. The return is never shown to the LLM.
2. **Extract** a free-form list of surprises (results vs. preview expectations) for every discovery event.

Extraction appends to `data/surprise/discovery_extractions.jsonl` one event at a time, so re-running a cell resumes where it stopped and retries failures.

In [16]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from suprise_dim import data
from suprise_dim import extraction

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Split

In [17]:
events = data.load_eligible_events(quarters=("2026Q3",))
events_by_id = {e["event_id"]: e for e in events}
print(f"{len(events)} eligible events")

1365 eligible events


In [18]:
# Creates the split once; afterwards it is loaded so the sample stays fixed.
# A split saved in the old discovery/holdout format (no "test" key) is rebuilt.
# Pass overwrite=True to save_split only if you really want a new sample.
split = data.load_split() if data.SPLIT_PATH.exists() else None
if split is None or "test" not in split:
    split = data.make_split(events, train_frac=0.8, n_discovery=500, seed=42)
    data.save_split(split, overwrite=True)

discovery = [events_by_id[eid] for eid in split["discovery"]]
print(
    f"{len(split['train'])} train, {len(split['test'])} test (from {split['test_start']}), "
    f"{len(discovery)} discovery sampled from train (seed {split['seed']})"
)

1076 train, 289 test (from 2026-08-13T21:00:00+00:00), 500 discovery sampled from train (seed 42)


In [19]:
# Sanity check: date ranges must not overlap, and return distributions should be similar.
def describe(ids):
    evs = [events_by_id[eid] for eid in ids]
    s = pd.Series([data.car1(e) for e in evs]).describe()
    s["first"] = min(e["event_datetime"] for e in evs)
    s["last"] = max(e["event_datetime"] for e in evs)
    return s

assert set(split["discovery"]) <= set(split["train"])
pd.DataFrame({name: describe(split[name]) for name in ("train", "test", "discovery")})

,train,test,discovery
count,1076.0,289.0,500.0
mean,0.004312,-0.021619,0.006602
std,0.136077,0.14258,0.142161
min,-0.604792,-0.550218,-0.604792
25%,-0.065902,-0.102891,-0.066656
50%,0.001946,-0.012022,0.00162
75%,0.064677,0.043232,0.062681
max,0.649427,0.738772,0.649427
first,2026-08-03T12:00:00+00:00,2026-08-13T21:00:00+00:00,2026-08-03T12:30:00+00:00
last,2026-08-13T20:30:00+00:00,2026-09-22T21:00:00+00:00,2026-08-13T20:30:00+00:00


## 2. Extraction

### Inspect one event

Look at the prompt and a single result before spending on the full run.

In [20]:
from suprise_dim.prompts import EXTRACTION_SYSTEM, build_user_message

sample_event = discovery[0]
print(EXTRACTION_SYSTEM)
print("\n" + "=" * 80 + "\n")
print(build_user_message(sample_event)[:3000])

You compare an earnings release against what was expected before it.

You receive:
1. Facts from the earnings call, written after the release.
2. A research preview written BEFORE the release: consensus, guidance, and the issues the market is watching.

List every distinct way the reported results, guidance, or commentary differ from what the preview led investors to expect. Cover everything: headline numbers, guidance, segments, margins, operating metrics, capital allocation, one-off items, management tone, and anything the preview flagged as a key debate. Also cover surprises the preview did not anticipate at all.

Rules:
- One item per distinct surprise. Do not merge unrelated surprises; do not split one into many.
- Describe each surprise concretely in plain language. Do not label it with an abstract category; the categories will be derived later from many releases.
- Measure every surprise against the expectation, not against the prior year. Strong growth that matched consensus is

In [ ]:
result = extraction.extract_event(sample_event)
for s in result.surprises:
    print(f"[{s.direction}/{s.magnitude}/{s.expectation_source}] {s.description}")
print("\nIn line:", *result.in_line, sep="\n- ")

### Run on the discovery set

Start with a small `N` as a smoke test, then set `N = None` for all 500.

In [ ]:
N = None  # None for the whole discovery set

summary = extraction.run_extraction(
    discovery[:N] if N else discovery,
    model=extraction.DEFAULT_MODEL,
    effort=extraction.DEFAULT_EFFORT,
    workers=8,
)
summary

## 3. Look at the results

In [21]:
# Only the current discovery sample; the file may hold extractions from an earlier split.
discovery_ids = set(split["discovery"])
records = [r for r in extraction.load_extractions() if r["event_id"] in discovery_ids]
items = pd.DataFrame([
    {"event_id": r["event_id"], "ticker": r["ticker"], "car1": r["car1"], **s}
    for r in records
    for s in r["surprises"]
])
per_event = items.groupby("event_id").size()
print(f"{len(records)} events, {len(items)} surprise items, {per_event.mean():.1f} per event on average")
items.head()

500 events, 3895 surprise items, 7.8 per event on average


,event_id,ticker,car1,description,direction,magnitude,fact_evidence,expectation_evidence,expectation_source
0,ea_QMCO_Q1_2027,QMCO,0.649427,"QMCO reported adjusted EBITDA of $8 million, f...",positive,large,"“Adjusted EBITDA was a positive $8 million, $5...",“Adj. EBITDA: $1.5M ± $1M.”,preview
1,ea_QMCO_Q1_2027,QMCO,0.649427,"Instead of the expected non-GAAP loss, QMCO re...",positive,large,“Non-GAAP net income was $4 million ($0.18 per...,“Non-GAAP EPS: ($0.15) ± $0.10”; consensus was...,preview
2,ea_QMCO_Q1_2027,QMCO,0.649427,"Revenue came in at $80.8 million, above both t...",positive,moderate,“Q1 FY2027 revenue was $80.8 million … exceedi...,“Consensus sits exactly on the guidance midpoi...,preview
3,ea_QMCO_Q1_2027,QMCO,0.649427,"GAAP gross margin reached 39.3%, above the pre...",positive,moderate,"“GAAP gross margin reached 39.3%, up 360 basis...",The preview calculated “~38% gross margin” imp...,preview
4,ea_QMCO_Q1_2027,QMCO,0.649427,Q2 revenue guidance is approximately $82 milli...,positive,moderate,“Q2 FY2027 revenue guidance is approximately $...,The preview said “a Q2 guide below ~$78M puts ...,preview


In [22]:
items[["direction", "magnitude", "expectation_source"]].apply(pd.Series.value_counts).fillna(0).astype(int)

,direction,magnitude,expectation_source
call,0,0,34
large,0,655,0
mixed,437,0,0
moderate,0,2424,0
negative,1079,0,0
none,0,0,680
positive,2318,0,0
preview,0,0,3181
small,0,816,0
unclear,61,0,0


In [23]:
# Rough check that the direction labels carry signal:
# net direction per event (positive minus negative items) vs. the realized return.
net = (
    items.assign(score=items["direction"].map({"positive": 1, "negative": -1}).fillna(0))
    .groupby("event_id")
    .agg(net_direction=("score", "sum"), car1=("car1", "first"))
)
# Spearman = Pearson on ranks; computed this way so it doesn't need scipy.
ranks = net[["net_direction", "car1"]].rank()
print("Spearman:", round(ranks["net_direction"].corr(ranks["car1"]), 3))
net.groupby("net_direction")["car1"].agg(["count", "mean", "median"])

Spearman: 0.465


,count,mean,median
net_direction,,,
-9.0,1,-0.132534,-0.132534
-8.0,1,-0.254943,-0.254943
-6.0,3,-0.125514,-0.006329
-5.0,8,-0.060753,-0.145945
-4.0,11,-0.142393,-0.133860
-3.0,15,-0.081926,-0.052462
-2.0,24,-0.054421,-0.055611
-1.0,38,-0.053351,-0.046273
0.0,40,-0.067787,-0.049265


In [24]:
# Read the raw items for one event.
eid = records[0]["event_id"]
for s in items[items.event_id == eid].itertuples():
    print(f"[{s.direction}/{s.magnitude}/{s.expectation_source}] {s.description}\n")

[positive/large/preview] QMCO reported adjusted EBITDA of $8 million, far above the preview’s $1.5 million guidance midpoint and $5.5 million above the guidance range’s high end. This was a much stronger profitability result than expected.

[positive/large/preview] Instead of the expected non-GAAP loss, QMCO reported $4 million of non-GAAP net income, or $0.18 per share. That is a substantial reversal from the preview’s expected loss per share.

[positive/moderate/preview] Revenue came in at $80.8 million, above both the $75 million guidance midpoint and roughly $75 million consensus. The preview characterized revenue as a low-differentiation variable centered on guidance, so the size of the beat was better than expected.

[positive/moderate/preview] GAAP gross margin reached 39.3%, above the preview’s roughly 38% margin implied by guidance and above its 37% validation threshold. The preview identified margin recovery as the key debate, so this was a stronger recovery than its central 

## 4. Group items into dimensions (step 3)

The loop:

1. **Propose** a taxonomy (v1) from all items of 80 random discovery events, starting from a textbook seed list (`prompts.SEED_DIMENSIONS`).
2. **Assign** every item to one dimension or `other`. The model can only answer with a key from the current taxonomy.
3. **Review** the counts, the `other` bucket and a sample per dimension.
4. **Refine** to the next version and reassign. Stop when `other` is small (under ~5%) and a refinement makes few changes.
5. **Check stability** by proposing from a different sample and measuring agreement.

Each version is saved to `data/surprise/taxonomy/` (`taxonomy_<version>.json`, `assignments_<version>.jsonl`). Assignment resumes like extraction does.

Proposal and refinement use `grouping.PROPOSE_MODEL`; assignment uses the cheaper `grouping.ASSIGN_MODEL` in batches of 40 items (about 100 calls per full pass).

In [25]:
from suprise_dim import grouping

items_df = grouping.surprise_items(records)
print(f"{len(items_df)} items from {items_df['event_id'].nunique()} events")

3895 items from 500 events


### 4a. Propose v1

In [26]:
if grouping.taxonomy_path("v1").exists():
    tax_v1 = grouping.load_taxonomy("v1")
else:
    tax_v1 = grouping.propose_taxonomy(items_df, n_events=80, seed=0)
    grouping.save_taxonomy(tax_v1, "v1")

for d in tax_v1.dimensions:
    print(f"{d.key:32s} {d.definition}")
print("\nChanges vs. seed list:", *tax_v1.changes, sep="\n- ")

forward_guidance                 Surprises in management's forecasts, targets, or outlook for future financial results or recurring operating KPIs, including their timing, qualifications, reaffirmation, or withdrawal. Covers company-wide, segment, and product-level outlooks.
revenue                          Surprises in recognized revenue, sales, or their growth rates for any reporting period. Includes consolidated, segment, geographic, product, and channel revenue, whether reported, organic, pro forma, or constant-currency.
operating_profitability          Surprises in reported gross profit, operating income, EBITDA and related profit measures, margins, or accounting profitability ratios. Applies equally to consolidated and business-unit results.
bottom_line_earnings             Surprises in reported earnings per share, net income or loss, or total pretax bottom-line earnings. Includes GAAP, adjusted, continuing-operations, and comparable distributable or operating earnings per share.

### 4b. Assign every item under v1

Try `items_df.head(200)` first to check the output, then run on all items.

In [28]:
grouping.assign_items(items_df, tax_v1, "v1")
asg_v1 = grouping.load_assignments("v1")
print(f"{len(asg_v1)}/{len(items_df)} items assigned")
asg_v1["dimension"].value_counts().to_frame("items").assign(share=lambda t: (t["items"] / t["items"].sum()).round(3))

Assigning 3695 items in 93 batches (200 already done) with gpt-6-luna...
[1/93] 40/40 assigned
[2/93] 40/40 assigned
[3/93] 40/40 assigned
[4/93] 40/40 assigned
[5/93] 40/40 assigned
[6/93] 40/40 assigned
[7/93] 40/40 assigned
[8/93] 40/40 assigned
[9/93] 40/40 assigned
[10/93] 40/40 assigned
[11/93] 40/40 assigned
[12/93] 40/40 assigned
[13/93] 40/40 assigned
[14/93] 40/40 assigned
[15/93] 40/40 assigned
[16/93] 40/40 assigned
[17/93] 40/40 assigned
[18/93] 40/40 assigned
[19/93] 40/40 assigned
[20/93] 40/40 assigned
[21/93] 40/40 assigned
[22/93] 40/40 assigned
[23/93] 40/40 assigned
[24/93] 40/40 assigned
[25/93] 40/40 assigned
[26/93] 40/40 assigned
[27/93] 40/40 assigned
[28/93] 40/40 assigned
[29/93] 40/40 assigned
[30/93] 40/40 assigned
[31/93] 40/40 assigned
[32/93] 40/40 assigned
[33/93] 40/40 assigned
[34/93] 40/40 assigned
[35/93] 40/40 assigned
[36/93] 40/40 assigned
[37/93] 40/40 assigned
[38/93] 40/40 assigned
[39/93] 40/40 assigned
[40/93] 40/40 assigned
[41/93] 40/40 as

,items,share
dimension,,
forward_guidance,660,0.169
revenue,584,0.150
operating_kpis,518,0.133
operating_profitability,460,0.118
bottom_line_earnings,266,0.068
cash_balance_sheet_liquidity,244,0.063
development_execution_regulatory,239,0.061
commercial_economics,232,0.060
capital_allocation,161,0.041


### 4c. Review

In [29]:
def review(asg, n_other=20, n_per_dim=3, seed=0):
    merged = items_df.merge(asg, on="item_id")
    other = merged[merged["dimension"] == grouping.OTHER]
    print(f"other: {len(other)} items ({len(other) / len(merged):.1%})\n")
    for text in other.sample(min(n_other, len(other)), random_state=seed)["description"]:
        print("-", text)
    for dim, rows in merged[merged["dimension"] != grouping.OTHER].groupby("dimension"):
        print(f"\n### {dim} ({len(rows)})")
        for text in rows.sample(min(n_per_dim, len(rows)), random_state=seed)["description"]:
            print("-", text)

review(asg_v1)

other: 1 items (0.0%)

- Duke reported $5 billion of economic-development investment and support for more than 9,000 jobs in the first half of 2026, an unanticipated positive operating and community-development update.

### accounting_tax_special_items (89)
- Owlet received about $4 million in tariff refunds, lifting Q2 gross margin and adjusted EBITDA: reported gross margin was 64.4% and adjusted EBITDA was $6.7 million including the refund, versus 54% and $2.9 million excluding it. The preview did not anticipate this one-time benefit.
- The tariff-refund benefit was much more concrete and material than the preview’s base case allowed for: Fortune Brands recognized $122 million gross, with $81 million flowing to operating income. It added 700 basis points to consolidated margin and $0.52 to quarterly EPS, helping lift reported operating margin to 20.4% versus the preview’s 13.0%–13.5% model.
- PRA increased European ERC by $349 million after a portfolio review, an unanticipated upward

### 4c-2. Reliability of the assignment

Reassign 400 random items under the same taxonomy, in shuffled batches (the original pass batched items event by event), and compare labels item by item. The rerun is saved as `assignments_v1_rerun_s0.jsonl`, about 10 calls.

- **agreement**: share of items given the same dimension both times.
- **kappa**: agreement beyond chance; roughly, above 0.8 is strong, 0.6–0.8 is acceptable, below 0.6 means the boundaries are too fuzzy for the labels to be trusted.
- **per_dimension**: how often items of each v1 dimension keep their label. Low values point to the boundaries the refinement should fix.
- **confusions**: the most common pairs of dimensions the model switches between.

Pass `model=...` to compare a different assigner, or a new `seed` for another sample.

In [30]:
rel = grouping.reliability_check(items_df, tax_v1, "v1", n_items=400, seed=0)
print(f"{rel['n']} items: agreement {rel['agreement']:.1%}, Cohen's kappa {rel['kappa']:.3f}")
display(rel["per_dimension"].round(2))
rel["confusions"].head(15)

Assigning 400 items in 10 batches (0 already done) with gpt-6-luna...
[1/10] 40/40 assigned
[2/10] 40/40 assigned
[3/10] 40/40 assigned
[4/10] 40/40 assigned
[5/10] 40/40 assigned
[6/10] 40/40 assigned
[7/10] 40/40 assigned
[8/10] 40/40 assigned
[9/10] 40/40 assigned
[10/10] 40/40 assigned
Done. 400 written, 0 missing, 0 batches failed; call again to retry.
400 items: agreement 90.8%, Cohen's kappa 0.896


,items,agreement
dimension_a,,
forward_guidance,65,0.89
revenue,64,0.95
operating_kpis,62,0.97
bottom_line_earnings,36,0.92
operating_profitability,34,0.97
commercial_economics,27,0.85
cash_balance_sheet_liquidity,24,1.00
development_execution_regulatory,24,0.79
capital_allocation,16,0.81


,dimension_a,dimension_b,items
0,bottom_line_earnings,operating_profitability,3
1,forward_guidance,operating_kpis,3
2,revenue,operating_kpis,2
3,accounting_tax_special_items,bottom_line_earnings,1
4,development_execution_regulatory,strategy_corporate_events,1
5,strategy_corporate_events,external_environment,1
6,revenue,commercial_economics,1
7,operating_profitability,accounting_tax_special_items,1
8,operating_kpis,revenue,1
9,operating_kpis,operating_profitability,1


### 4d. Refine and reassign

Set `PREV` / `NEXT` and rerun this cell for each round (v1 → v2 → v3 …). `agreement` is the normalized mutual information between the two versions' assignments: close to 1 means the refinement barely changed how items are grouped.

In [31]:
PREV, NEXT = "v1", "v2"

tax_prev, asg_prev = grouping.load_taxonomy(PREV), grouping.load_assignments(PREV)
if grouping.taxonomy_path(NEXT).exists():
    tax_next = grouping.load_taxonomy(NEXT)
else:
    tax_next = grouping.refine_taxonomy(tax_prev, items_df, asg_prev)
    grouping.save_taxonomy(tax_next, NEXT)
print(f"{PREV} -> {NEXT} changes:", *tax_next.changes, sep="\n- ")

grouping.assign_items(items_df, tax_next, NEXT)
asg_next = grouping.load_assignments(NEXT)

nmi, table = grouping.agreement(asg_prev, asg_next)
print(f"\nAgreement {PREV} vs {NEXT}: NMI = {nmi:.3f}")
print(f"other: {PREV} {(asg_prev['dimension'] == grouping.OTHER).mean():.1%} -> {NEXT} {(asg_next['dimension'] == grouping.OTHER).mean():.1%}")
table

v1 -> v2 changes:
- Reordered dimensions by assigned frequency, retaining counts only as ordering evidence rather than estimating reassigned totals.
- Extended operating_kpis to cover recurring company-supported economic-development investment and jobs, absorbing the lone other item; distinguished facilitated activity from the company's own capital deployment.
- Tightened operating_kpis and revenue boundaries: recognized product or brand sales belong to revenue, sales penetration and business mix to commercial_economics, and aggregate awards to operating_kpis.
- Clarified forward_guidance coverage of future savings, business-mix targets, and quantified financial contributions, while distinguishing current funded runway from future financial targets.
- Clarified cash_balance_sheet_liquidity coverage of current funding sufficiency, principal recoveries, and early leverage-target attainment; separated balances from repayment actions and asset-sale agreements.
- Tightened development_execu

dimension_b,accounting_tax_special_items,bottom_line_earnings,capital_allocation,cash_balance_sheet_liquidity,commercial_economics,costs_expenses,development_execution_regulatory,disclosure_delivery,external_environment,financing_capital_structure,forward_guidance,operating_kpis,operating_profitability,other,revenue,strategy_corporate_events
dimension_a,,,,,,,,,,,,,,,,
accounting_tax_special_items,80,4,0,2,0,1,0,0,1,0,0,1,0,0,0,0
bottom_line_earnings,0,259,0,0,0,0,0,0,0,0,1,0,3,0,3,0
capital_allocation,0,0,139,1,2,1,0,0,0,1,6,2,0,0,0,9
cash_balance_sheet_liquidity,1,0,1,238,0,0,0,0,0,1,0,2,0,1,0,0
commercial_economics,0,0,2,0,179,0,4,0,3,0,13,28,0,0,2,1
costs_expenses,1,0,1,0,1,136,0,0,2,0,6,0,1,0,0,0
development_execution_regulatory,1,0,0,0,4,1,200,1,4,2,6,12,0,0,3,5
disclosure_delivery,2,0,0,1,0,0,0,14,0,0,0,0,0,0,0,1
external_environment,0,0,0,0,2,1,0,0,70,0,7,4,0,0,1,0


In [32]:
review(asg_next)

other: 1 items (0.0%)

- Book value per share was $36, below the preview’s approximate $37 expectation. The company still reported 8.5% year-over-year growth, but the reported level was lower than the preview anticipated.

### accounting_tax_special_items (92)
- The quarter included about $100 million of net positive mark-to-market and timing impacts, but management said the first half still had a carried net negative position that would mostly reverse in Q3. That makes the Q2 timing benefit real but less complete or earlier than the preview’s discussion of a Q2 reversal might have led investors to expect.
- The quarter included $8.6 million of 45Z credits, slightly above the preview’s expected $5–8 million contribution. The preview’s range was close, so the excess was modest rather than a major departure.
- A $9.9 million unrealized gain on a minority equity investment was recorded in other income, an unanticipated gain that increased reported income but was not a recurring operating 

### 4e. Stability

Propose a taxonomy from a different sample of events (`seed=1`), assign 1,000 items under it, and compare with your current version on those items. A low NMI means the dimensions depend on which events happened to be sampled; read the crosstab to see which ones disagree.

In [33]:
CURRENT = "v2"
ALT = "alt_seed1"

if grouping.taxonomy_path(ALT).exists():
    tax_alt = grouping.load_taxonomy(ALT)
else:
    tax_alt = grouping.propose_taxonomy(items_df, n_events=80, seed=1)
    grouping.save_taxonomy(tax_alt, ALT)

check_items = items_df.sample(1000, random_state=0)
grouping.assign_items(check_items, tax_alt, ALT)

nmi, table = grouping.agreement(grouping.load_assignments(CURRENT), grouping.load_assignments(ALT))
print(f"Agreement {CURRENT} vs {ALT}: NMI = {nmi:.3f}")
table

Assigning 1000 items in 25 batches (0 already done) with gpt-6-luna...
[1/25] 40/40 assigned
[2/25] 40/40 assigned
[3/25] 40/40 assigned
[4/25] 40/40 assigned
[5/25] 39/40 assigned
[6/25] 40/40 assigned
[7/25] 40/40 assigned
[8/25] 40/40 assigned
[9/25] 40/40 assigned
[10/25] 40/40 assigned
[11/25] 40/40 assigned
[12/25] 40/40 assigned
[13/25] 40/40 assigned
[14/25] 40/40 assigned
[15/25] 40/40 assigned
[16/25] 40/40 assigned
[17/25] 40/40 assigned
[18/25] 40/40 assigned
[19/25] 40/40 assigned
[20/25] 40/40 assigned
[21/25] 40/40 assigned
[22/25] 40/40 assigned
[23/25] 40/40 assigned
[24/25] 40/40 assigned
[25/25] 40/40 assigned
Done. 999 written, 1 missing, 0 batches failed; call again to retry.
Agreement v2 vs alt_seed1: NMI = 0.640


dimension_b,accounting_tax_special_items,balance_sheet_financing,capital_allocation,cash_flow,commercial_agreements_access,corporate_strategy_transactions,costs_efficiency,demand_customer_activity,disclosure_management_communication,earnings,operating_execution_capacity,operating_profitability,other,pricing_monetization_mix,product_development_regulation,revenue
dimension_a,,,,,,,,,,,,,,,,
accounting_tax_special_items,20,0,0,0,0,0,1,0,1,0,0,2,0,0,0,0
bottom_line_earnings,2,0,0,0,0,0,0,0,0,79,0,2,0,0,0,0
capital_allocation,1,1,30,0,0,0,0,0,0,0,0,0,0,0,0,0
cash_balance_sheet_liquidity,0,35,1,23,0,0,0,0,0,0,0,0,0,0,0,0
commercial_economics,0,0,1,0,24,0,0,2,0,0,0,0,0,16,1,0
costs_expenses,2,0,0,0,1,0,35,0,0,0,2,2,0,1,0,0
development_execution_regulatory,1,0,0,0,3,0,0,2,0,0,8,0,0,0,45,1
disclosure_delivery,0,0,0,0,0,0,0,1,2,0,0,0,0,0,0,0
external_environment,0,0,0,0,0,0,1,7,2,0,0,0,0,2,6,0


## 5. Extend to train_rest and test

Run the same two steps as discovery on the other events, with the taxonomy frozen at **v2**:

- **train_rest**: the 576 train events outside the discovery sample.
- **test**: the 289 test events. Extract and assign them now, but do not look at their returns until the model is fixed.

Each set gets its own files: `data/surprise/<set>_extractions.jsonl` and `data/surprise/taxonomy/assignments_v2_<set>.jsonl`. Both resume like before. Extraction uses `extraction.DEFAULT_MODEL` at `DEFAULT_EFFORT`, the same as discovery; assignment uses `grouping.ASSIGN_MODEL`.

Cost: about 865 extraction calls plus about 170 assignment batches.

In [34]:
from suprise_dim import dataset

FROZEN = "v2"
tax = grouping.load_taxonomy(FROZEN)
sets = dataset.event_sets(split, events_by_id)
print({name: len(evs) for name, evs in sets.items()})
print("discovery extraction model(s):", {r["model"] for r in extraction.load_extractions()}, "| now:", extraction.DEFAULT_MODEL, extraction.DEFAULT_EFFORT)

{'discovery': 500, 'train_rest': 576, 'test': 289}
discovery extraction model(s): {'gpt-6-luna'} | now: gpt-6-luna medium


### 5a. Extract surprises

Set `N` to a small number first as a smoke test, then `None`.

In [36]:
N = None  # None for every event in the set

for name in ("train_rest", "test"):
    evs = sets[name][:N] if N else sets[name]
    extraction.run_extraction(evs, out_path=data.extractions_path(name))

Extracting 566 events (10 already done) with gpt-6-luna...
[1/566] ea_HSIC_Q2_2026: 6 surprises
[2/566] ea_SHLS_Q2_2026: 8 surprises
[3/566] ea_FWRG_Q2_2026: 10 surprises
[4/566] ea_RYTM_Q2_2026: 10 surprises
[5/566] ea_LDOS_Q2_2026: 9 surprises
[6/566] ea_TREX_Q2_2026: 8 surprises
[7/566] ea_WAT_Q2_2026: 10 surprises
[8/566] ea_DOCN_Q2_2026: 8 surprises
[9/566] ea_ANDE_Q2_2026: 3 surprises
[10/566] ea_WHR_Q2_2026: 7 surprises
[11/566] ea_AHCO_Q2_2026: 10 surprises
[12/566] ea_APO_Q2_2026: 10 surprises
[13/566] ea_W_Q2_2026: 12 surprises
[14/566] ea_BALL_Q2_2026: 12 surprises
[15/566] ea_BR_Q4_2026: 8 surprises
[16/566] ea_BRBR_Q3_2026: 10 surprises
[17/566] ea_CAT_Q2_2026: 11 surprises
[18/566] ea_CERT_Q2_2026: 7 surprises
[19/566] ea_PVLA_Q2_2026: 7 surprises
[20/566] ea_NPO_Q2_2026: 8 surprises
[21/566] ea_SEAT_Q2_2026: 6 surprises
[22/566] ea_IDXX_Q2_2026: 7 surprises
[23/566] ea_XMTR_Q2_2026: 6 surprises
[24/566] ea_AIN_Q2_2026: 8 surprises
[25/566] ea_ZBRA_Q2_2026: 8 surprises
[2

### 5b. Assign every surprise to a v2 dimension

In [37]:
for name in ("train_rest", "test"):
    ids = {e["event_id"] for e in sets[name]}
    recs = [r for r in extraction.load_extractions(data.extractions_path(name)) if r["event_id"] in ids]
    grouping.assign_items(grouping.surprise_items(recs), tax, dataset.assignment_version(FROZEN, name))

Assigning 4448 items in 112 batches (0 already done) with gpt-6-luna...
[1/112] 40/40 assigned
[2/112] 40/40 assigned
[3/112] 40/40 assigned
[4/112] 40/40 assigned
[5/112] 40/40 assigned
[6/112] 40/40 assigned
[7/112] 40/40 assigned
[8/112] 40/40 assigned
[9/112] 40/40 assigned
[10/112] 40/40 assigned
[11/112] 40/40 assigned
[12/112] 40/40 assigned
[13/112] 40/40 assigned
[14/112] 40/40 assigned
[15/112] 40/40 assigned
[16/112] 40/40 assigned
[17/112] 40/40 assigned
[18/112] 40/40 assigned
[19/112] 40/40 assigned
[20/112] 40/40 assigned
[21/112] 40/40 assigned
[22/112] 40/40 assigned
[23/112] 40/40 assigned
[24/112] 40/40 assigned
[25/112] 40/40 assigned
[26/112] 40/40 assigned
[27/112] 40/40 assigned
[28/112] 40/40 assigned
[29/112] 40/40 assigned
[30/112] 40/40 assigned
[31/112] 40/40 assigned
[32/112] 40/40 assigned
[33/112] 40/40 assigned
[34/112] 40/40 assigned
[35/112] 40/40 assigned
[36/112] 40/40 assigned
[37/112] 40/40 assigned
[38/112] 40/40 assigned
[39/112] 40/40 assigned
[

### 5c. Combined table

One row per surprise across discovery, train_rest and test, saved to `data/surprise/surprises_v2.csv`. The checks below are descriptive only; no returns are used.

In [42]:
surprises = dataset.build_surprise_frame(split, FROZEN)
surprises.to_csv(dataset.surprises_path(FROZEN), index=False)

events_with_rows = set(surprises["event_id"])
summary = pd.DataFrame({
    name: {
        "events in set": len(evs),
        "events extracted": len({r["event_id"] for r in extraction.load_extractions(data.extractions_path(name))} & {e["event_id"] for e in evs}),
        "events with no surprise": sum(e["event_id"] not in events_with_rows for e in evs),
        "surprises": int((surprises["split"] == name).sum()),
        "unassigned": int(((surprises["split"] == name) & surprises["dimension"].isna()).sum()),
        "other": int(((surprises["split"] == name) & (surprises["dimension"] == "other")).sum()),
    }
    for name, evs in sets.items()
})
display(summary)
surprises.head()

,discovery,train_rest,test
events in set,500,576,289
events extracted,500,576,289
events with no surprise,0,0,18
surprises,3895,4448,2188
unassigned,0,0,0
other,1,3,1


,split,event_id,ticker,event_datetime,car1,earnings_surprise,item_id,position,n_items,dimension,dimension_name,direction,magnitude,expectation_source,description,fact_evidence,expectation_evidence,taxonomy_version,extraction_model
0,discovery,ea_QMCO_Q1_2027,QMCO,2026-08-10T21:00:00+00:00,0.649427,0.03056,ea_QMCO_Q1_2027#0,0,8,operating_profitability,"Operating profit, margins, and returns",positive,large,preview,"QMCO reported adjusted EBITDA of $8 million, f...","“Adjusted EBITDA was a positive $8 million, $5...",“Adj. EBITDA: $1.5M ± $1M.”,v2,gpt-6-luna
1,discovery,ea_QMCO_Q1_2027,QMCO,2026-08-10T21:00:00+00:00,0.649427,0.03056,ea_QMCO_Q1_2027#1,1,8,bottom_line_earnings,EPS and bottom-line earnings,positive,large,preview,"Instead of the expected non-GAAP loss, QMCO re...",“Non-GAAP net income was $4 million ($0.18 per...,“Non-GAAP EPS: ($0.15) ± $0.10”; consensus was...,v2,gpt-6-luna
2,discovery,ea_QMCO_Q1_2027,QMCO,2026-08-10T21:00:00+00:00,0.649427,0.03056,ea_QMCO_Q1_2027#2,2,8,revenue,Reported revenue and sales,positive,moderate,preview,"Revenue came in at $80.8 million, above both t...",“Q1 FY2027 revenue was $80.8 million … exceedi...,“Consensus sits exactly on the guidance midpoi...,v2,gpt-6-luna
3,discovery,ea_QMCO_Q1_2027,QMCO,2026-08-10T21:00:00+00:00,0.649427,0.03056,ea_QMCO_Q1_2027#3,3,8,operating_profitability,"Operating profit, margins, and returns",positive,moderate,preview,"GAAP gross margin reached 39.3%, above the pre...","“GAAP gross margin reached 39.3%, up 360 basis...",The preview calculated “~38% gross margin” imp...,v2,gpt-6-luna
4,discovery,ea_QMCO_Q1_2027,QMCO,2026-08-10T21:00:00+00:00,0.649427,0.03056,ea_QMCO_Q1_2027#4,4,8,forward_guidance,Financial and operating outlook,positive,moderate,preview,Q2 revenue guidance is approximately $82 milli...,“Q2 FY2027 revenue guidance is approximately $...,The preview said “a Q2 guide below ~$78M puts ...,v2,gpt-6-luna


## 6. Event-level features

One row per event (1,076 train, 289 test); discovery and train_rest are both labelled **train**. Built from `surprises_v2.csv` by `features.build_event_features` and saved to `data/surprise/features_v2.csv`.

- **Per dimension (45):** `net_w__<dim>` = Σ direction × magnitude (±1 × 1/2/3; mixed and unclear count 0), and its positive and negative parts `pos_w__<dim>` / `neg_w__<dim>` (net = pos − neg, so don't feed all three to an unregularized model).
- **Totals (9):** item counts by direction, `net`, `net_w`, the same over the first three items, `net_per_item`.
- **Flags and options (5):** `no_surprises` (extraction found nothing: 18 test events whose call facts are only procedural; no train event), and option-implied volatility, absolute move and 25-delta skew with `has_option_stats` (NaN when absent, about 42% coverage).
- **Target:** `target_pct`, car1's rank among all 2,362 events of 2026Q3, scaled to [0, 1].

Only train rows are inspected here; test stays untouched until the model is fixed.

In [43]:
from suprise_dim import features

surprises = pd.read_csv(dataset.surprises_path(FROZEN))
feat = features.build_event_features(surprises, split, events_by_id, tax)
feat.to_csv(features.features_path(FROZEN), index=False)
print(feat.shape, feat["split"].value_counts().to_dict())

train_feat = feat[feat["split"] == "train"]
r = train_feat.rank(numeric_only=True)
print("train Spearman with target_pct:",
      {c: round(r[c].corr(r["target_pct"]), 3) for c in ["net", "net_w", "net_first3", "net_w_first3", "net_per_item"]})
feat.head()

(1365, 71) {'train': 1076, 'test': 289}
train Spearman with target_pct: {'net': np.float64(0.443), 'net_w': np.float64(0.468), 'net_first3': np.float64(0.444), 'net_w_first3': np.float64(0.466), 'net_per_item': np.float64(0.433)}


,event_id,ticker,event_datetime,split,car1,target_pct,earnings_surprise,net_w__forward_guidance,net_w__revenue,net_w__operating_kpis,...,no_surprises,opt_implied_earnings_volatility,opt_implied_absolute_earnings_move,opt_skew_25_delta,has_option_stats,baseline__deepseek,baseline__gemini,baseline__moonshotai,baseline__openai,baseline__z-ai
0,ea_TWST_Q3_2026,TWST,2026-08-03T12:00:00+00:00,train,0.088587,0.840745,-0.003495,8,4,0,...,0,0.1529,0.1219,NaN,1,0.88,0.85,0.80,0.88,0.62
1,ea_ABTC_Q2_2026,ABTC,2026-08-03T12:30:00+00:00,train,0.044492,0.725964,-0.172101,0,0,4,...,0,0.1287,0.1026,NaN,1,0.80,0.78,0.55,0.81,0.58
2,ea_KRYS_Q2_2026,KRYS,2026-08-03T12:30:00+00:00,train,-0.098985,0.158831,-0.000293,0,-2,1,...,0,NaN,NaN,NaN,0,0.85,0.82,0.62,0.82,0.82
3,ea_MAR_Q2_2026,MAR,2026-08-03T12:30:00+00:00,train,-0.085027,0.187632,0.000376,-3,1,3,...,0,0.0441,0.0352,0.022,1,0.85,0.65,0.80,0.86,0.58
4,ea_SBH_Q3_2026,SBH,2026-08-03T12:30:00+00:00,train,0.062198,0.781872,0.001336,0,-3,-2,...,0,NaN,NaN,NaN,0,0.60,0.52,0.60,0.82,0.58
